In [9]:
! pip install streamlit sentence-transformers pdfplumber python-docx
!pip install streamlit
!pip install docx2txt
import streamlit as st
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import pdfplumber
import docx2txt
# Load SBERT model
model = SentenceTransformer("all-MiniLM-L6-v2")
# --- File Readers ---
def extract_text_from_pdf(uploaded_file):
  with pdfplumber.open(uploaded_file) as pdf:
    return " ".join([page.extract_text()
    for page in pdf.pages if page.extract_text()])
    def extract_text_from_docx(uploaded_file):
      return docx2txt.process(uploaded_file)
      # --- Soft Skills Matching ---
      soft_skill_prompts = [ "teamwork", "communication", "leadership", "problem solving", "volunteering", "critical thinking", "time management", "collaboration" ]
      def compute_soft_skill_score(resume_text):
        scores = []
        for prompt in soft_skill_prompts:
          sim = util.cos_sim( model.encode(resume_text, convert_to_tensor=True), model.encode(prompt, convert_to_tensor=True) ).item()
          scores.append(sim)
          return sum(scores) / len(scores)
          # --- Semantic Similarity Matching ---
          def compute_semantic_fit_score(resume_text, job_desc_list):
            resume_emb = model.encode(resume_text, convert_to_tensor=True)
            job_embs = model.encode(job_desc_list.tolist(), convert_to_tensor=True)
            similarities = util.cos_sim(resume_emb, job_embs).squeeze().cpu().numpy()
            return similarities
            # --- Resume to Job Matching Pipeline ---
            def match_resume_to_jobs(resume_text, df_jobs):
              job_descs = df_jobs["Job Desc"]
              semantic_scores = compute_semantic_fit_score(resume_text, job_descs)
              soft_skill_score = compute_soft_skill_score(resume_text)
              final_scores = []
              for i in range(len(df_jobs)):
                final = 0.8 * semantic_scores[i] + 0.2 * soft_skill_score
                final_scores.append(f"{round(final * 100, 2)}%")
                df_jobs["Skill Fit Score (%)"] = final_scores
                return df_jobs.sort_values(by="Skill Fit Score (%)", ascending=False)
                # --- Streamlit UI ---
                st.set_page_config(page_title="Semantic Job Matcher", layout="centered")
                st.title("🎯 Semantic Resume-to-Job Matching")
                st.markdown("Upload your **resume** (PDF or DOCX) and a **CSV job dataset** to find your best-fit roles.")
                uploaded_resume = st.file_uploader("Upload your Resume", type=["pdf", "docx"])
                uploaded_csv = st.file_uploader("Upload Job Dataset (CSV)", type=["csv"])
                if uploaded_resume and uploaded_csv:
                  # Extract resume text
                  if uploaded_resume.name.endswith(".pdf"):
                    resume_text = extract_text_from_pdf(uploaded_resume)
                  elif uploaded_resume.name.endswith(".docx"):
                      resume_text = extract_text_from_docx(uploaded_resume)
                  else: st.error("Unsupported resume format.")
                  st.stop()
                  df = pd.read_csv(uploaded_csv)
                      # Filter for relevant roles
                  df_filtered = df[df["Job Title"].str.contains("analyst|data|bi|engineer|consultant", case=False)]
                      # Run matching
                  result_df = match_resume_to_jobs(resume_text, df_filtered)
                      # Display top matches
                  st.success("✅ Matching complete. Top roles below:")
                  st.dataframe(result_df[["Job Title", "Company Name", "Job Location", "Skill Fit Score (%)"]].head(10))

  Using cached docx2txt-0.9-py3-none-any.whl.metadata (529 bytes)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]